In [76]:
from langgraph.graph import StateGraph, MessagesState,START, END,add_messages
from langchain_core.messages import BaseMessage,HumanMessage,AIMessage
from typing import TypedDict,Annotated,List,Sequence
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyPDFLoader
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pinecone import ServerlessSpec,Pinecone
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough,RunnableLambda
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import os
from langgraph.prebuilt import ToolNode,tools_condition

In [77]:
load_dotenv()

True

In [78]:
pc=Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index_name = "practice-rag" 

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        serverless=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(index_name)

In [79]:
llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0.7
)

In [80]:
class RAGState(TypedDict):
    messages:Annotated[List[BaseMessage],add_messages]

In [81]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [82]:
rag_prompt = PromptTemplate.from_template(
   template= """You are a helpful assistant.
Use only the provided context to answer the question.
If the answer is not present in the context, say: "I don't know based on the provided context."

Context:
{context}

Question:
{question}

Answer:"""
)

In [83]:
file_path = r"D:\ProdRAG\prodRAG\Blockchain_Course_Proposal.pdf"
loader = PyPDFLoader(file_path)
documents = loader.load()

In [84]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
texts = text_splitter.split_documents(documents)

In [85]:
embedder = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview",output_dimensionality=384)


In [86]:
vectorstore = PineconeVectorStore.from_documents(
    texts,
    embedder,
    index_name=index_name
)

In [87]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [99]:
query="What is module 5?"

In [100]:
result = retriever.invoke(query)

context = [doc.page_content for doc in result]
metadata = [doc.metadata for doc in result]

In [101]:
print(context)


['blockchain technology. \nModule 2: Cryptography Fundamentals — Basics of hashing, public and private key \nencryption, and digital signatures. \nModule 3: Consensus Algorithms — Understanding Proof of Work, Proof of Stake, and \nByzantine Fault Tolerance. \nModule 4: Smart Contracts and Ethereum — Writing and deploying smart contracts using \nSolidity and Remix IDE. \nModule 5: Blockchain and AI Integration — Using blockchain to secure AI model sharing \nand training data provenance.', 'blockchain technology. \nModule 2: Cryptography Fundamentals — Basics of hashing, public and private key \nencryption, and digital signatures. \nModule 3: Consensus Algorithms — Understanding Proof of Work, Proof of Stake, and \nByzantine Fault Tolerance. \nModule 4: Smart Contracts and Ethereum — Writing and deploying smart contracts using \nSolidity and Remix IDE. \nModule 5: Blockchain and AI Integration — Using blockchain to secure AI model sharing \nand training data provenance.', 'blockchain tec

In [102]:
print(metadata)

[{'author': 'python-docx', 'creationdate': '2025-10-10T09:37:45+05:30', 'creator': 'Microsoft® Word LTSC', 'moddate': '2025-10-10T09:37:45+05:30', 'page': 0.0, 'page_label': '1', 'producer': 'Microsoft® Word LTSC', 'source': 'D:\\ProdRAG\\prodRAG\\Blockchain_Course_Proposal.pdf', 'total_pages': 4.0}, {'author': 'python-docx', 'creationdate': '2025-10-10T09:37:45+05:30', 'creator': 'Microsoft® Word LTSC', 'moddate': '2025-10-10T09:37:45+05:30', 'page': 0.0, 'page_label': '1', 'producer': 'Microsoft® Word LTSC', 'source': 'D:\\ProdRAG\\prodRAG\\Blockchain_Course_Proposal.pdf', 'total_pages': 4.0}, {'author': 'python-docx', 'creationdate': '2025-10-10T09:37:45+05:30', 'creator': 'Microsoft® Word LTSC', 'moddate': '2025-10-10T09:37:45+05:30', 'page': 0.0, 'page_label': '1', 'producer': 'Microsoft® Word LTSC', 'source': 'D:\\ProdRAG\\prodRAG\\Blockchain_Course_Proposal.pdf', 'total_pages': 4.0}]


In [ ]:
['blockchain technology. \nModule 2: Cryptography Fundamentals — Basics of hashing, public and private key \nencryption, and digital signatures. \nModule 3: Consensus Algorithms — Understanding Proof of Work, Proof of Stake, and \nByzantine Fault Tolerance. \nModule 4: Smart Contracts and Ethereum — Writing and deploying smart contracts using \nSolidity and Remix IDE. \nModule 5: Blockchain and AI Integration — Using blockchain to secure AI model sharing \nand training data provenance.', 'blockchain technology. \nModule 2: Cryptography Fundamentals — Basics of hashing, public and private key \nencryption, and digital signatures. \nModule 3: Consensus Algorithms — Understanding Proof of Work, Proof of Stake, and \nByzantine Fault Tolerance. \nModule 4: Smart Contracts and Ethereum — Writing and deploying smart contracts using \nSolidity and Remix IDE. \nModule 5: Blockchain and AI Integration — Using blockchain to secure AI model sharing \nand training data provenance.', 'blockchain technology. \nModule 2: Cryptography Fundamentals — Basics of hashing, public and private key \nencryption, and digital signatures. \nModule 3: Consensus Algorithms — Understanding Proof of Work, Proof of Stake, and \nByzantine Fault Tolerance. \nModule 4: Smart Contracts and Ethereum — Writing and deploying smart contracts using \nSolidity and Remix IDE. \nModule 5: Blockchain and AI Integration — Using blockchain to secure AI model sharing \nand training data provenance.']


In [ ]:
[{'author': 'python-docx', 'creationdate': '2025-10-10T09:37:45+05:30', 'creator': 'Microsoft® Word LTSC', 'moddate': '2025-10-10T09:37:45+05:30', 'page': 0.0, 'page_label': '1', 'producer': 'Microsoft® Word LTSC', 'source': 'D:\\ProdRAG\\prodRAG\\Blockchain_Course_Proposal.pdf', 'total_pages': 4.0}, {'author': 'python-docx', 'creationdate': '2025-10-10T09:37:45+05:30', 'creator': 'Microsoft® Word LTSC', 'moddate': '2025-10-10T09:37:45+05:30', 'page': 0.0, 'page_label': '1', 'producer': 'Microsoft® Word LTSC', 'source': 'D:\\ProdRAG\\prodRAG\\Blockchain_Course_Proposal.pdf', 'total_pages': 4.0}, {'author': 'python-docx', 'creationdate': '2025-10-10T09:37:45+05:30', 'creator': 'Microsoft® Word LTSC', 'moddate': '2025-10-10T09:37:45+05:30', 'page': 0.0, 'page_label': '1', 'producer': 'Microsoft® Word LTSC', 'source': 'D:\\ProdRAG\\prodRAG\\Blockchain_Course_Proposal.pdf', 'total_pages': 4.0}]

In [ ]:
# def rag_tool(query):

#     """
#     Retrieve relevant information from the pdf document.
#     Use this tool when the user asks factual / conceptual questions
#     that might be answered from the stored documents.
#     """
#     result = retriever.invoke(query)

#     context = [doc.page_content for doc in result]
#     metadata = [doc.metadata for doc in result]

#     return {
#         'query': query,
#         'context': context,
#         'metadata': metadata
#     }

In [ ]:
@tool
def rag_tool(query):
    '''A tool that takes a user query, retrieves relevant information from the PDF, and returns an answer based on the retrieved context.
    Use this tool when the user asks factual/conceptual questions that might be answered from the stored documents.'''
    
    # Retrieve similar chunks from the vectorstore
    docs = retriever.invoke(query)
    context = format_docs(docs)
    
    # Generate answer using the context
    rag_chain = RunnableParallel({
        "question": RunnablePassthrough(),
        "context": RunnableLambda(lambda x: context)
    })
    
    output = (rag_chain | rag_prompt | llm | StrOutputParser()).invoke(query)
    
    return {
        "query": query,
        "output": output
    }

In [129]:
tools=[rag_tool]
llm_with_tools=llm.bind_tools(tools)

In [130]:
def chat_rag(state:RAGState):
    response=llm_with_tools.invoke(state["messages"])
    return {"messages":[response]}

In [131]:
tool_node=ToolNode(tools)

In [132]:
workflow=StateGraph(RAGState)
workflow.add_node("chat_rag",chat_rag)
workflow.add_node("tools", tool_node)
workflow.add_edge(START, "chat_rag")
workflow.add_conditional_edges(
    "chat_rag",
    tools_condition,
    {"tools": "tools",END: END}

)
workflow.add_edge("tools", "chat_rag")      

app=workflow.compile()

In [136]:
result=app.invoke({
"messages":[HumanMessage(content=("What is module1?"))]
    })

In [137]:
print(result["messages"][-1].content)

I'm sorry, but it seems I am unable to find the information you're looking for.
